<a href="https://colab.research.google.com/github/jioh-shin/econ5200-lab01-data-portfolio1/blob/main/replication_starter_card1995_(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Replication Starter: Card (1995), Proximity to College and the Return to Schooling
## ECON 5200 · Midterm Replication Study · instrumental variables

Card, D. (1995). "Using Geographic Variation in College Proximity to Estimate the Return to Schooling." In
*Aspects of Labour Market Behaviour*, ed. Christofides, Grant and Swidinsky. University of Toronto Press.

> **AI rule for this project.** Sections 1 and 2 (positioning, direct replication) are **no AI**: not for
> the code, not for the prose. Sections 3 to 6 allow AI as a documented co-pilot under P.R.I.M.E., with the full
> transcript in `ai-appendix.md`. Every cell below is marked with the section it belongs to.

**How to use this notebook.** It is a starter, not the project. Copy it into your repository made from
[RyanPiao/econ5200-replication-template](https://github.com/RyanPiao/econ5200-replication-template), run it top to
bottom, and fill the cells marked **YOUR TURN**. Cells marked **GIVEN** run as they are; cells marked **CHECK** test
what you wrote and print a message, never the answer. A blank copy runs from top to bottom and prints no estimate:
the numbers appear once you have written Section 2.

In [ ]:
# GIVEN — setup: the IV package Colab does not have, then the libraries
%pip install linearmodels -q

import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

print("Setup complete.")

Setup complete.


## The data

`card.dta` is the NLS Young Men cohort as Card used it: men aged 14 to 24 in 1966, interviewed again in 1976. One row
per man. The variables the design uses:

| variable | what it is |
|---|---|
| `lwage` | log hourly wage, 1976 (the outcome) |
| `educ` | years of schooling completed by 1976 (the endogenous treatment) |
| `nearc4` | 1 if he grew up near a four-year college in 1966 (the instrument) |
| `nearc2` | 1 if near a two-year college in 1966 (a second instrument the paper discusses) |
| `exper`, `expersq` | potential experience and its square |
| `black`, `south`, `smsa` | race, and region and urban residence in 1976 |
| `reg661`–`reg669`, `smsa66` | region and urban residence in 1966 |

Read the paper's table notes before you choose controls: which of these does the specification you replicate
include, and which year's region variables?

In [ ]:
# GIVEN — Section 2 starts here: load the data
card = pd.read_stata("https://raw.githubusercontent.com/scunning1975/mixtape/master/card.dta")
print(f"{len(card):,} men, {card.shape[1]} columns")
card[["lwage", "educ", "nearc4", "nearc2", "exper", "black", "south", "smsa"]].describe().round(3)

3,010 men, 37 columns


,lwage,educ,nearc4,nearc2,exper,black,south,smsa
count,3010.000,3010.000,3010.000,3010.000,3010.000,3010.000,3010.000,3010.000
mean,6.262,13.263,0.682,0.441,8.856,0.234,0.404,0.713
std,0.444,2.677,0.466,0.497,4.142,0.423,0.491,0.452
min,4.605,1.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,5.977,12.000,0.000,0.000,6.000,0.000,0.000,0.000
50%,6.287,13.000,1.000,0.000,8.000,0.000,0.000,1.000
75%,6.564,16.000,1.000,1.000,11.000,0.000,1.000,1.000
max,7.785,18.000,1.000,1.000,23.000,1.000,1.000,1.000


## Section 2: direct replication — YOUR TURN, no AI

Write the two-stage least squares estimate of the return to schooling that the paper reports as its headline:
`lwage` on `educ`, instrumented by `nearc4`, with the controls the paper's table uses. Use `IV2SLS` from
`linearmodels` (`IV2SLS(dependent, exog, endog, instruments).fit(...)`; `sm.add_constant` builds `exog`).

Set two names, because Section 3 reuses them:

- `CONTROLS`: the list of control column names you include
- `iv_fit`: your fitted `IV2SLS` result
- `ols_fit`: the OLS regression of `lwage` on `educ` and the same controls (`sm.OLS`), the estimate IV is compared with

Then print the coefficient on `educ` and its standard error. Choose the covariance option the paper's standard
errors correspond to, and say which in your report.

In [ ]:
# YOUR TURN — Section 2, no AI: define CONTROLS, iv_fit and ols_fit, then print the estimate on educ

CONTROLS = ["exper", "expersq", "black", "south", "smsa", "smsa66", "reg662", "reg663", "reg664", "reg665", "reg666", "reg667", "reg668", "reg669"]

# OLS
X_ols = sm.add_constant(card[["educ"] + CONTROLS])
ols_fit = sm.OLS(card["lwage"], X_ols).fit()

# IV / 2SLS
X_exog = sm.add_constant(card[CONTROLS])

iv_fit = IV2SLS(dependent=card["lwage"], exog=X_exog, endog=card[["educ"]], instruments=card[["nearc4"]]).fit(cov_type="unadjusted", debiased=True)

print("OLS:", ols_fit.params["educ"])
print("OLS SE:", ols_fit.bse["educ"])

print("IV:", iv_fit.params["educ"])
print("IV SE:", iv_fit.std_errors["educ"])


OLS: 0.07469324651520436
OLS SE: 0.0034983455789416224
IV: 0.13150377777765243
IV SE: 0.054963667584744656


In [ ]:
# GIVEN — Section 2: your replication table's first rows, OLS beside 2SLS, and the first-stage F
if iv_fit is None or ols_fit is None:
    print("Fill Section 2 first: iv_fit and ols_fit are not set yet.")
else:
    fs = sm.OLS(card["educ"], sm.add_constant(card[["nearc4"] + CONTROLS])).fit()
    print(f"{'':<8}{'return to a year':>18}{'SE':>9}")
    print(f"{'OLS':<8}{ols_fit.params['educ']:>18.4f}{ols_fit.bse['educ']:>9.4f}")
    print(f"{'2SLS':<8}{iv_fit.params['educ']:>18.4f}{iv_fit.std_errors['educ']:>9.4f}")
    print(f"first stage on nearc4: {fs.params['nearc4']:+.3f} years, F = {fs.tvalues['nearc4'] ** 2:.1f}")

          return to a year       SE
OLS                 0.0747   0.0035
2SLS                0.1315   0.0550
first stage on nearc4: +0.320 years, F = 13.3


### Your number, with its unit

`lwage` is a natural log, so the coefficient on `educ` is a change in the log wage per year of schooling. Times
100 it is in **log points** (0.10 is 10 log points), which is approximately the percent change in the wage;
100·(e^b − 1) is the exact percent. Both readings pass under "percent". It is not a percentage point: that unit belongs to
outcomes that are themselves proportions. Fill all five names:

- `PRINTED`: the coefficient exactly as your code prints it
- `IN_UNITS`: the same estimate in the unit you name in `UNIT`
- `UNIT`: one of `log points per year of schooling`, `percent per year of schooling`
- `PAPER_NUMBER`, `PAPER_WHERE`: the paper's own estimate, and its table, column and row, read from the paper

In [ ]:
# YOUR TURN — Section 2, no AI
PRINTED = float(iv_fit.params["educ"])
IN_UNITS = 100 * PRINTED
UNIT = "log points per year of schooling"
PAPER_NUMBER = 0.132
PAPER_WHERE = "Table 4, Row 1, IV estimate"

In [ ]:
# CHECK — Section 2: your headline number, with its unit. Run it; you need not read it.
def check_headline(printed, in_units, unit, paper_number, paper_where, units_ok):
    """Is the unit right for this outcome, and is the in-unit number the printed one converted?"""
    if any(v is None for v in (printed, in_units, unit, paper_number, paper_where)):
        print("Not filled yet: set all five names in the cell above.")
        return False
    u = " ".join(str(unit).lower().split())
    if u not in units_ok:
        print(f"'{unit}' is not a unit this outcome is measured in. Allowed here: " + "; ".join(sorted(units_ok)) + ".")
        print("Ask what the outcome variable is: a log, a proportion (0 to 1), or a rate per 100,000.")
        return False
    wants = [f(float(printed)) for f in units_ok[u]]     # every accepted reading of this unit
    if not any(abs(float(in_units) - w) <= max(0.005, 0.02 * abs(w)) for w in wants):
        print(f"Your in-unit number does not follow from your printed estimate in '{unit}'.")
        print("Convert the printed coefficient again: what does one unit of the outcome mean here?")
        return False
    if not str(paper_where).strip() or len(str(paper_where).split()) < 2:
        print("Name where the paper's number is: table, column (or figure), and row.")
        return False
    print("Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.")
    return True

CARD_UNITS = {
    "log points per year of schooling": [lambda b: 100 * b],   # a log point is 100 x the change in the log
    "percent per year of schooling": [lambda b: 100 * b, lambda b: 100 * (np.exp(b) - 1)],   # approximation, or exact
}
ok = check_headline(PRINTED, IN_UNITS, UNIT, PAPER_NUMBER, PAPER_WHERE, CARD_UNITS)

Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.


## Section 3: robustness — GIVEN code, AI allowed from here

Each function takes your Section 2 choices, so the numbers below are about *your* specification. Run them, then
write what each one implies about the original claim. That third part is what is graded.

**1. The first stage, and how strong it is.** The first-stage F with one instrument is the instrument's t
statistic squared. The rule of thumb is F above 10; a valid conventional 5% t test needs about 104.7 (Lee,
McCrary, Moreira and Porter 2022). The heteroskedasticity-robust F is printed beside the classical one.

In [ ]:
# GIVEN — Section 3: the first stage for any controls, instrument and sample
def first_stage(controls, instrument="nearc4", data=None):
    data = card if data is None else data
    X = sm.add_constant(data[[instrument] + controls])
    classical = sm.OLS(data["educ"], X).fit()
    robust = sm.OLS(data["educ"], X).fit(cov_type="HC1")
    return {"coef": classical.params[instrument], "F": classical.tvalues[instrument] ** 2,
            "F_robust": robust.tvalues[instrument] ** 2, "n": int(classical.nobs)}


if CONTROLS is None:
    print("Fill Section 2 first: CONTROLS is not set yet.")
else:
    fs = first_stage(CONTROLS)
    print(f"First stage on nearc4: {fs['coef']:+.3f} years of schooling, "
          f"F = {fs['F']:.1f} (robust {fs['F_robust']:.1f}), n = {fs['n']:,}")

First stage on nearc4: +0.320 years of schooling, F = 13.3 (robust 14.1), n = 3,010


**2. A weak-instrument-robust interval.** Anderson–Rubin never divides by the first stage: for each candidate
return b0 it asks whether `lwage − b0·educ` is still related to the instrument, and keeps every b0 it cannot
reject. If the set runs into the edge of the grid, it is unbounded there, and the grid is not the answer.

In [ ]:
# GIVEN — Section 3: Anderson–Rubin 95% set over a grid of candidate returns
def anderson_rubin(controls, instrument="nearc4", grid=np.linspace(-0.2, 0.6, 801), data=None):
    data = card if data is None else data
    X = sm.add_constant(data[[instrument] + controls])
    keep = [b0 for b0 in grid
            if sm.OLS(data["lwage"] - b0 * data["educ"], X).fit(cov_type="HC1").pvalues[instrument] > 0.05]
    if not keep:
        return None
    edge = keep[0] == grid[0] or keep[-1] == grid[-1]
    return min(keep), max(keep), edge


if CONTROLS is None:
    print("Fill Section 2 first: CONTROLS is not set yet.")
else:
    ar = anderson_rubin(CONTROLS)
    if ar is None:
        print("No value on the grid survives: widen the grid.")
    else:
        lo, hi, edge = ar
        wald = iv_fit.conf_int().loc["educ"]
        print(f"Anderson-Rubin 95% set: [{lo:.3f}, {hi:.3f}]" + ("  (touches the grid edge: unbounded there)" if edge else ""))
        print(f"Your Section 2 Wald interval: [{wald.iloc[0]:.3f}, {wald.iloc[1]:.3f}]")

Anderson-Rubin 95% set: [0.029, 0.281]
Your Section 2 Wald interval: [0.024, 0.239]


**3. Your own specification.** Change at least one choice the paper made and justify it: a different control set
(family background, the 1966 versus 1976 region variables), the other instrument (`nearc2`), or a subsample (for
example outside the South, or by race). `MY_SPEC` is a dictionary the code below understands:
`{"controls": [...], "instrument": "nearc4" or "nearc2", "sample": None or a pandas query string}`.

In [ ]:
# YOUR TURN — Section 3: your specification, and two sentences on why it is defensible
MY_SPEC = {"controls": CONTROLS, "instrument": "nearc2", "sample": None}
MY_SPEC_WHY = ("I replace proximity to a four-year college with proximity " "to a two-year college as the instrument for schooling. " "This tests whether the estimated return to education " "depends on the particular measure of college accessibility.")

In [ ]:
# CHECK — Section 3: your own specification. Run it; you need not read it.
def check_my_spec(my_spec, baseline, why):
    if my_spec is None or why is None:
        print("Not filled yet: set MY_SPEC and MY_SPEC_WHY above.")
        return False
    if my_spec == baseline:
        print("MY_SPEC is the baseline. Change at least one choice, and say why that choice is defensible.")
        return False
    sentences = [s for s in str(why).replace("?", ".").split(".") if len(s.split()) >= 5]
    if len(sentences) < 2:
        print("MY_SPEC_WHY needs at least two sentences: what you changed, and why a careful author might have made that choice.")
        return False
    print("Specification accepted. Report it beside the baseline and say what the difference implies about the original claim.")
    return True

BASELINE = None if CONTROLS is None else {"controls": CONTROLS, "instrument": "nearc4", "sample": None}
if check_my_spec(MY_SPEC, BASELINE, MY_SPEC_WHY):
    d = card.query(MY_SPEC["sample"]) if MY_SPEC["sample"] else card
    z, c = MY_SPEC["instrument"], MY_SPEC["controls"]
    n_before = len(d)
    d = d.dropna(subset=["lwage", "educ", z] + c)             # family background has missing values; both fits need the same rows
    if len(d) < n_before:
        print(f"Dropped {n_before - len(d):,} rows with a missing value in your variables: n = {len(d):,} of {n_before:,}")
    dropped = [x for x in c if d[x].nunique() < 2]           # a control that cannot vary in this sample
    c = [x for x in c if x not in dropped]
    if dropped:
        print("Dropped, constant in this sample:", ", ".join(dropped))
    fit = IV2SLS(d["lwage"], sm.add_constant(d[c]), d["educ"], d[z]).fit(cov_type=iv_fit.cov_type)
    fs = first_stage(c, z, d)
    print(f"Your specification: 2SLS {fit.params['educ']:.4f} (SE {fit.std_errors['educ']:.4f}), "
          f"first-stage F {fs['F']:.1f}, n = {fs['n']:,}")
    print(f"Section 2 baseline: 2SLS {iv_fit.params['educ']:.4f} (SE {iv_fit.std_errors['educ']:.4f})")

Specification accepted. Report it beside the baseline and say what the difference implies about the original claim.
Your specification: 2SLS 0.2932 (SE 0.1849), first-stage F 2.5, n = 3,010
Section 2 baseline: 2SLS 0.1315 (SE 0.0550)


**4. Your other exercises.** Section 3 asks for at least four. Natural ones on this paper: the first stage and
2SLS in subsamples where the exclusion restriction is more or less plausible; `nearc2` and `nearc4` together, and
what an over-identification test can and cannot tell you; adding family background (`fatheduc`, `motheduc`,
`momdad14`), and what the rows lost to missing values do to the sample; a placebo outcome that proximity should not
move. Write them below this cell.

In [ ]:
family_controls = ["fatheduc", "motheduc", "momdad14"]
extended_controls = CONTROLS + family_controls

required = ["lwage", "educ", "nearc4"] + extended_controls
df_family = card.dropna(subset=required).copy()

X_base = sm.add_constant(df_family[CONTROLS])
X_family = sm.add_constant(df_family[extended_controls])

iv_base_same = IV2SLS(dependent=df_family["lwage"], exog=X_base, endog=df_family[["educ"]], instruments=df_family[["nearc4"]]).fit(cov_type="unadjusted", debiased=True)

iv_family = IV2SLS(dependent=df_family["lwage"], exog=X_family, endog=df_family[["educ"]], instruments=df_family[["nearc4"]]).fit(cov_type="unadjusted", debiased=True)

print("Original sample:", len(card))
print("Family-controls sample:", len(df_family))
print("\nOriginal baseline IV:", round(iv_fit.params["educ"], 4))
print("Baseline IV on restricted sample:", round(iv_base_same.params["educ"], 4))
print("IV with family controls:", round(iv_family.params["educ"], 4))
print("Family-controls SE:", round(iv_family.std_errors["educ"], 4))


Original sample: 3010
Family-controls sample: 2220

Original baseline IV: 0.1315
Baseline IV on restricted sample: 0.0799
IV with family controls: 0.0829
Family-controls SE: 0.0768


In [ ]:
results = []

for name, value in [("South", 1), ("Non-South", 0)]:
    subset = card.loc[card["south"] == value].copy()

    # Drop controls that are constant in this subsample
    varying = [c for c in CONTROLS if subset[c].nunique() > 1]

    X = sm.add_constant(subset[varying], has_constant="add")

    iv = IV2SLS(dependent=subset["lwage"], exog=X, endog=subset[["educ"]], instruments=subset[["nearc4"]]).fit(cov_type="unadjusted", debiased=True)

    fs = sm.OLS(subset["educ"], sm.add_constant(subset[["nearc4"] + varying], has_constant="add")).fit()

    results.append({"Region": name, "N": len(subset), "IV estimate": round(iv.params["educ"], 4), "SE": round(iv.std_errors["educ"], 4), "First-stage F": round(fs.tvalues["nearc4"] ** 2, 2)})

regional_results = pd.DataFrame(results)
print(regional_results.to_string(index=False))

   Region    N  IV estimate     SE  First-stage F
    South 1215       0.3264 0.1831           2.95
Non-South 1795       0.0555 0.0568          11.47


In [ ]:
# Preliminary comparison only: specifications differ from the original paper
comparison = pd.DataFrame({
    "Estimate": ["OLS coefficient", "OLS SE", "IV coefficient", "IV SE"], "Paper": [0.073, 0.004, 0.132, 0.049], "Replication": [ols_fit.params["educ"], ols_fit.bse["educ"], iv_fit.params["educ"], iv_fit.std_errors["educ"]]})
comparison["Gap"] = (comparison["Replication"] - comparison["Paper"])

print(comparison.round(4).to_string(index=False))

       Estimate  Paper  Replication     Gap
OLS coefficient  0.073       0.0747  0.0017
         OLS SE  0.004       0.0035 -0.0005
 IV coefficient  0.132       0.1315 -0.0005
          IV SE  0.049       0.0550  0.0060


In [ ]:

# Section 3: Two Instruments and Overidentification Test

# Keep the same controls and sample as the baseline
X = sm.add_constant(card[CONTROLS])

# Use both college proximity measures as instruments
iv_two = IV2SLS(dependent=card["lwage"], exog=X, endog=card[["educ"]], instruments=card[["nearc4", "nearc2"]]).fit(cov_type="unadjusted", debiased=True)

# First-stage regression with both instruments
first_stage_two = sm.OLS(card["educ"], sm.add_constant(card[CONTROLS + ["nearc4", "nearc2"]])).fit()

# Joint test of both instruments
joint_test = first_stage_two.f_test("nearc4 = 0, nearc2 = 0")

# Sargan overidentification test
sargan = iv_two.sargan

print("Baseline IV:", round(iv_fit.params["educ"], 4))
print("Two-instrument IV:", round(iv_two.params["educ"], 4))
print("Two-instrument SE:", round(iv_two.std_errors["educ"], 4))
print("Joint first-stage F:", round(float(joint_test.fvalue), 2))
print("Sargan statistic:", round(sargan.stat, 4))
print("Sargan p-value:", round(sargan.pval, 4))


Baseline IV: 0.1315
Two-instrument IV: 0.1571
Two-instrument SE: 0.0526
Joint first-stage F: 7.89
Sargan statistic: 1.2482
Sargan p-value: 0.2639


In [ ]:
print(card.columns.tolist())

['id', 'nearc2', 'nearc4', 'educ', 'age', 'fatheduc', 'motheduc', 'weight', 'momdad14', 'sinmom14', 'step14', 'reg661', 'reg662', 'reg663', 'reg664', 'reg665', 'reg666', 'reg667', 'reg668', 'reg669', 'south66', 'black', 'smsa', 'south', 'smsa66', 'wage', 'enroll', 'KWW', 'IQ', 'married', 'libcrd14', 'exper', 'lwage', 'expersq', 'u_lwage', 'lwage_hat', 'u_educ']


In [ ]:
# Section 3: Placebo / Covariate Balance Test
# Check whether college proximity predicts family structure at age 14

placebo_controls = [c for c in CONTROLS if c not in ["exper", "expersq"]]

placebo_vars = ["momdad14", "nearc4"] + placebo_controls
placebo_df = card[placebo_vars].dropna().copy()

X_placebo = sm.add_constant(placebo_df[["nearc4"] + placebo_controls])

placebo_fit = sm.OLS(placebo_df["momdad14"], X_placebo).fit(cov_type="HC3")

print("Placebo outcome: momdad14")
print("Sample size:", int(placebo_fit.nobs))
print("Coefficient on nearc4:",
      round(placebo_fit.params["nearc4"], 4))
print("HC3 standard error:",
      round(placebo_fit.bse["nearc4"], 4))
print("p-value:",
      round(placebo_fit.pvalues["nearc4"], 4))
print("95% confidence interval:",
      placebo_fit.conf_int().loc["nearc4"].round(4).tolist())

Placebo outcome: momdad14
Sample size: 3010
Coefficient on nearc4: -0.0215
HC3 standard error: 0.0181
p-value: 0.2342
95% confidence interval: [-0.057, 0.0139]


In [ ]:
variables_to_check = ["age", "educ", "exper", "expersq", "fatheduc", "motheduc", "momdad14", "sinmom14", "step14", "nearc4", "nearc2"]

print(card[variables_to_check].isna().sum())
print("\nVariable summary:")
print(card[variables_to_check].describe().round(3))

age           0
educ          0
exper         0
expersq       0
fatheduc    690
motheduc    353
momdad14      0
sinmom14      0
step14        0
nearc4        0
nearc2        0
dtype: int64

Variable summary:
            age      educ     exper   expersq  fatheduc  motheduc  momdad14  \
count  3010.000  3010.000  3010.000  3010.000  2320.000  2657.000  3010.000   
mean     28.120    13.263     8.856    95.579    10.003    10.348     0.789   
std       3.137     2.677     4.142    84.618     3.721     3.180     0.408   
min      24.000     1.000     0.000     0.000     0.000     0.000     0.000   
25%      25.000    12.000     6.000    36.000     8.000     8.000     1.000   
50%      28.000    13.000     8.000    64.000    10.000    12.000     1.000   
75%      31.000    16.000    11.000   121.000    12.000    12.000     1.000   
max      34.000    18.000    23.000   529.000    18.000    18.000     1.000   

       sinmom14    step14    nearc4    nearc2  
count  3010.000  3010.000  3010.

In [ ]:
# Section 2: Intermediate IV specification (not exact replication)
# Treat education, experience, and experience squared as endogenous

card_exact = card.copy()
card_exact["agesq"] = card_exact["age"] ** 2

# Exclude endogenous variables from exogenous controls
EXOG_CONTROLS = [c for c in CONTROLS if c not in ["exper", "expersq"]]

X_exog = sm.add_constant(card_exact[EXOG_CONTROLS])

# Three endogenous regressors
X_endog = card_exact[["educ", "exper", "expersq"]]

# Three excluded instruments
Z = card_exact[["nearc4", "age", "agesq"]]

iv_extended = IV2SLS(dependent=card_exact["lwage"], exog=X_exog, endog=X_endog, instruments=Z).fit(cov_type="unadjusted", debiased=True)

print("Extended IV results")
print("N:", iv_extended.nobs)
print("Education coefficient:",
      round(iv_extended.params["educ"], 4))
print("Education SE:",
      round(iv_extended.std_errors["educ"], 4))

print("\nPaper Table 4 Row 1:")
print("IV coefficient: 0.132")
print("IV standard error: 0.049")

Extended IV results
N: 3010
Education coefficient: 0.1224
Education SE: 0.0465

Paper Table 4 Row 1:
IV coefficient: 0.132
IV standard error: 0.049


In [ ]:
# Prepare parental education variables
# Preserve the original sample using mean imputation

card_exact["fatheduc_missing"] = (card_exact["fatheduc"].isna().astype(int))

card_exact["motheduc_missing"] = (card_exact["motheduc"].isna().astype(int))

card_exact["fatheduc_imp"] = (card_exact["fatheduc"].fillna(card_exact["fatheduc"].mean()))

card_exact["motheduc_imp"] = (card_exact["motheduc"].fillna(card_exact["motheduc"].mean()))

print("Original N:", len(card_exact))
print("Missing father education:",
      card_exact["fatheduc_missing"].sum())
print("Missing mother education:",
      card_exact["motheduc_missing"].sum())

print("Remaining missing values:", card_exact[["fatheduc_imp", "motheduc_imp"]].isna().sum().sum())

Original N: 3010
Missing father education: 690
Missing mother education: 353
Remaining missing values: 0


In [ ]:
# Section 2: Extended IV model with family background controls
# Approximate specification, not an exact replication of Card (1995)

family_vars = ["fatheduc_imp", "motheduc_imp", "fatheduc_missing", "motheduc_missing", "momdad14", "sinmom14"]

full_controls = EXOG_CONTROLS + family_vars

X_full = sm.add_constant(card_exact[full_controls])

iv_family_extended = IV2SLS(dependent=card_exact["lwage"], exog=X_full, endog=card_exact[["educ", "exper", "expersq"]], instruments=card_exact[["nearc4", "age", "agesq"]]).fit(cov_type="unadjusted", debiased=True)

print("Sample size:", iv_family_extended.nobs)
print("IV coefficient:",
      round(iv_family_extended.params["educ"], 4))
print("Standard error:",
      round(iv_family_extended.std_errors["educ"], 4))

print("Paper coefficient: 0.132")
print("Paper standard error: 0.049")

# Matching OLS specification for the extended IV model

X_ols_matched = sm.add_constant(card_exact[["educ", "exper", "expersq"] + full_controls])

ols_matched = sm.OLS(card_exact["lwage"], X_ols_matched).fit()

print("Matched OLS coefficient:",
      round(ols_matched.params["educ"], 4))
print("Matched OLS SE:",
      round(ols_matched.bse["educ"], 4))
print("Extended IV coefficient:",
      round(iv_family_extended.params["educ"], 4))
print("Extended IV SE:",
      round(iv_family_extended.std_errors["educ"], 4))


Sample size: 3010
IV coefficient: 0.1281
Standard error: 0.0486
Paper coefficient: 0.132
Paper standard error: 0.049
Matched OLS coefficient: 0.0723
Matched OLS SE: 0.0037
Extended IV coefficient: 0.1281
Extended IV SE: 0.0486


The baseline model and the extended IV model above are not exact replications of Table 4 in Card (1995). I will compare the variables and estimation method with the original paper before using the results for the direct replication.

In [ ]:
print("Original family variables:")
print(family_vars)

print("\nOther controls:")
print(EXOG_CONTROLS)

print("\nFamily structure variables:")
print(card_exact[["momdad14", "sinmom14", "step14"]].sum())

Original family variables:
['fatheduc_imp', 'motheduc_imp', 'fatheduc_missing', 'motheduc_missing', 'momdad14', 'sinmom14']

Other controls:
['black', 'south', 'smsa', 'smsa66', 'reg662', 'reg663', 'reg664', 'reg665', 'reg666', 'reg667', 'reg668', 'reg669']

Family structure variables:
momdad14    2376.0
sinmom14     303.0
step14       117.0
dtype: float32


## Sections 4 to 6, in your repository

- **Section 4, does the finding hold?** Use your Section 3 results. "Holds in a narrower population", "holds at a
  smaller magnitude" and "holds under an assumption the authors did not state" are all answers.
- **Section 5, peer audit.** Assigned on the date the brief gives, never on your own paper; file the issue by the
  project deadline in the brief's Checkpoints list.
- **Section 6, your own repository.** Move the code you keep from this notebook into the template's layout, pin
  versions in `requirements.txt` (this notebook was built with pandas 2.2.2, numpy 2.0.2, statsmodels 0.14.6,
  linearmodels 7.1, rdrobust 2.0.0 and rddensity 3.0), and make the README name the one command that reproduces
  your headline table. Then clone your own repository into an empty folder and run that command.